In [2]:
import pandas as pd
import requests as req
from bs4 import BeautifulSoup as B, NavigableString, Tag, Comment, Doctype
import time

In [ ]:


# links = []
# titles = []
# for i in range(1, 206):
#     url = f'https://search.ltn.com.tw/list?keyword=%E8%98%87%E8%8A%B1%E5%85%AC%E8%B7%AF&start_time=20041201&end_time=20250617&sort=date&type=all&page={i}'
#     print(f'抓取第 {i} 頁：{url}')

#     resp = req.get(url)
#     if resp.status_code != 200 :
#         print('Error status_code')
#         continue

#     soup = B(resp.text, 'html.parser')
#     sites = soup.find('ul', class_='list boxTitle')
#     for li_ in sites.find_all('li'):
#         a_tag = li_.find('a', class_='ph', href=True)
#         if a_tag :
#             link = a_tag['href']
#             title = a_tag['title']
#             links.append(link)
#             titles.append(title)
#     time.sleep(1)

# data = pd.DataFrame({'Link': links, 'Title': titles})
# data.to_csv('中時_蘇花公路.csv', index=False)

In [6]:
data = pd.read_csv('中時_蘇花公路.csv')
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4090 entries, 0 to 4089
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   Link    4090 non-null   object
 1   Title   4090 non-null   object
dtypes: object(2)
memory usage: 64.0+ KB


In [7]:
data.shape

(4090, 2)

In [27]:
data['Context'] = ''; data['PublishTime'] = ''

failed_urls = []

for i, url in enumerate(data['Link']):
    print(f'[{i+1}/{len(data)}] 處理中：{url}')

    try:
        resp = req.get(url)

        if resp.status_code != 200:
            print(f'無法存取：{resp.status_code}')
            data.at[i, 'Context'] = ''
            data.at[i, 'PublishTime'] = ''
            failed_urls.append(url)
            continue

        soup = B(resp.text, 'html.parser')

        time_tag = soup.find('span', class_='time')
        publish_time = time_tag.get_text(strip=True) if time_tag else ''
        data.at[i, 'PublishTime'] = publish_time

        article_span = soup.find_all('p')
        context = []
    
        for p in article_span:
            if p.get('style') and 'display:none' in p.get('style'):
                continue
            if p.find_parent(id='checkIE') or p.find_parent(class_='suggest') or p.find_parent(class_='photo boxTitle') or p.find_parent(class_='image-popup-vertical-fit') or p.find_parent(class_="appE1121"):
                continue
            EXCLUDE_P_CLASSES = {'appE1121', 'photoCaption', 'ga_event', 'copyright', 'before_ir'}
            if 'class' in p.attrs and any(cls in EXCLUDE_P_CLASSES for cls in p['class']):
                continue
            if p.find_parent(class_='ltnfooter boxTitle') or p.find_parent(class_='see_more boxTitle') or p.find_parent(class_='idle_con boxTitle'):
                continue
            if p.find_parent(id='right_blake') or p.find_parent(class_='Boom'):
                continue
            text = p.get_text(strip=True)
            if text:
                context.append(text)
        data.at[i, 'Context'] = '\n'.join(context)
        
    except Exception as e:
        print(f'發生錯誤:{e}')
        data.at[i, 'Context'] = ''
        data.at[i, 'PublishTime'] = ''
        failed_urls.append(url)

    data.to_csv('中時_蘇花公路_temp.csv', index=False)
    time.sleep(1) 

data.to_csv('中時_蘇花公路_full.csv', index=False)
with open('中時_蘇花公路_failed.txt', 'w', encoding='utf-8') as f:
    for url in failed_urls:
        f.write(url+'\n')
print('全部完成, 已儲存完整資料和失敗網址')

[1/4090] 處理中：https://news.ltn.com.tw/news/life/breakingnews/5076759
[2/4090] 處理中：https://news.ltn.com.tw/news/life/breakingnews/5072962
[3/4090] 處理中：https://news.ltn.com.tw/news/society/breakingnews/5068513
[4/4090] 處理中：https://news.ltn.com.tw/news/life/breakingnews/5067946
[5/4090] 處理中：https://news.ltn.com.tw/news/life/paper/1709174
[6/4090] 處理中：https://news.ltn.com.tw/news/life/breakingnews/5058655
[7/4090] 處理中：https://news.ltn.com.tw/news/life/paper/1708550
[8/4090] 處理中：https://news.ltn.com.tw/news/politics/breakingnews/5050873
[9/4090] 處理中：https://news.ltn.com.tw/news/life/paper/1707946
[10/4090] 處理中：https://news.ltn.com.tw/news/life/paper/1707947
[11/4090] 處理中：https://news.ltn.com.tw/news/politics/breakingnews/5049763
[12/4090] 處理中：https://news.ltn.com.tw/news/life/paper/1707784
[13/4090] 處理中：https://news.ltn.com.tw/news/politics/breakingnews/5049472
[14/4090] 處理中：https://news.ltn.com.tw/news/life/breakingnews/5049300
[15/4090] 處理中：https://news.ltn.com.tw/news/life/breakingnews/50

KeyboardInterrupt: 